In [1]:
# Cell 1: Imports & config

import os
import sys
import numpy as np
import pandas as pd

import torch
import torch.backends.cudnn as cudnn

# Disable CuDNN RNNs to avoid weird LSTM backward issues (we saw this before)
cudnn.enabled = False

import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel
from pytorch_forecasting.metrics import MAE

pl.seed_everything(42, workers=True)

print("Torch version     :", torch.__version__)
print("Lightning version :", pl.__version__)

DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"
print("Data path:", DATA_PATH)


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
Seed set to 42


Torch version     : 2.5.1+cu121
Lightning version : 2.5.6
Data path: C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet


In [2]:
# Cell 2: Load parquet and inspect

df = pd.read_parquet(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns  :", df.columns.tolist())

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

df.head()


Raw shape: (61473, 14)
Columns  : ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245


In [3]:
# Cell 3: Basic structure (close, time features, series_id, time_idx)

# Use mid close as main price
df["close"] = df["mid_c"]

# Time features as categoricals
df["hour"] = df["time"].dt.hour.astype(str)
df["day_of_week"] = df["time"].dt.dayofweek.astype(str)

# Single series id
df["series_id"] = "eurusd"

# Integer time index (1 per hour)
df["time_idx"] = np.arange(len(df), dtype=int)

df[["time", "close", "hour", "day_of_week", "series_id", "time_idx"]].head()


,time,close,hour,day_of_week,series_id,time_idx
0,2016-01-07 00:00:00+00:00,1.07778,0,3,eurusd,0
1,2016-01-07 01:00:00+00:00,1.08029,1,3,eurusd,1
2,2016-01-07 02:00:00+00:00,1.08152,2,3,eurusd,2
3,2016-01-07 03:00:00+00:00,1.08187,3,3,eurusd,3
4,2016-01-07 04:00:00+00:00,1.08236,4,3,eurusd,4


In [5]:
# Cell 4: Use your indicators.py + build 4h target

project_root = r"C:\Users\admin\Desktop\Forex_algo\code"
if project_root not in sys.path:
    sys.path.append(project_root)

from technicals.indicators import BollingerBands, ATR, KeltnerChannels, RSI, MACD

print("Using indicators from:", project_root)

# 1) Apply your indicators (on mid_* prices)
df = BollingerBands(df)        # BB_MA, BB_UP, BB_LW
df = ATR(df, n=14)             # ATR_14
df = KeltnerChannels(df)       # EMA, KeUp, KeLo
df = RSI(df, n=14)             # RSI_14
df = MACD(df)                  # MACD, SIGNAL, HIST

# 2) Create standard alias names for model (avoid duplicates)
df["rsi"] = df["RSI_14"]

df["macd"]        = df["MACD"]
df["macd_signal"] = df["SIGNAL"]
df["macd_hist"]   = df["HIST"]

df["bb_middle"] = df["BB_MA"]
df["bb_upper"]  = df["BB_UP"]
df["bb_lower"]  = df["BB_LW"]

df["atr14"] = df["ATR_14"]

# 3) Extra EMAs on mid_c
df["ema_5"]   = df["mid_c"].ewm(span=5,   min_periods=5).mean()
df["ema_20"]  = df["mid_c"].ewm(span=20,  min_periods=20).mean()
df["ema_50"]  = df["mid_c"].ewm(span=50,  min_periods=50).mean()
df["ema_200"] = df["mid_c"].ewm(span=200, min_periods=200).mean()

# 4) Engineered price features
df["momentum_oc"]       = df["mid_o"] - df["mid_c"]
df["avg_price_hl"]      = (df["mid_h"] + df["mid_l"]) / 2.0
df["range_hl"]          = df["mid_h"] - df["mid_l"]
df["typical_price_ohlc"] = (df["mid_o"] + df["mid_h"] + df["mid_l"] + df["mid_c"]) / 4.0

# 5) Log volume
df["log_volume"] = np.log1p(df["volume"])

# 6) 4-hour target: log-return between t and t+4 (aggregated 4h move)
HORIZON = 4  # hours
df["target_return_4h"] = np.log(df["close"].shift(-HORIZON)) - np.log(df["close"])

# Drop NaNs from rolling windows and last 4 entries from shift
df = df.dropna().reset_index(drop=True)

print("Shape after indicators & 4h target:", df.shape)
df[["time", "close", "target_return_4h"]].head(10)


Using indicators from: C:\Users\admin\Desktop\Forex_algo\code
Shape after indicators & 4h target: (61270, 48)


,time,close,target_return_4h
0,2016-01-19 07:00:00+00:00,1.08652,-0.000203
1,2016-01-19 08:00:00+00:00,1.08846,-0.000919
2,2016-01-19 09:00:00+00:00,1.08724,-0.000570
3,2016-01-19 10:00:00+00:00,1.08730,0.001682
4,2016-01-19 11:00:00+00:00,1.08630,0.003208
5,2016-01-19 12:00:00+00:00,1.08746,0.002727
6,2016-01-19 13:00:00+00:00,1.08662,0.005616
7,2016-01-19 14:00:00+00:00,1.08913,0.003071
8,2016-01-19 15:00:00+00:00,1.08979,0.002401
9,2016-01-19 16:00:00+00:00,1.09043,0.000559


In [6]:
# Cell 5: Curated feature list

FEATURE_COLS_EXT = [
    # Core prices & volume
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume", "log_volume",

    # Your indicators (standard names)
    "rsi",
    "macd", "macd_signal", "macd_hist",
    "atr14",
    "bb_lower", "bb_middle", "bb_upper",
    "ema_5", "ema_20", "ema_50", "ema_200",

    # Engineered
    "momentum_oc", "avg_price_hl", "range_hl", "typical_price_ohlc",
]

print("FEATURE_COLS_EXT:")
print(FEATURE_COLS_EXT)


FEATURE_COLS_EXT:
['mid_o', 'mid_h', 'mid_l', 'mid_c', 'volume', 'log_volume', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'atr14', 'bb_lower', 'bb_middle', 'bb_upper', 'ema_5', 'ema_20', 'ema_50', 'ema_200', 'momentum_oc', 'avg_price_hl', 'range_hl', 'typical_price_ohlc']


In [7]:
# Cell 6: Time-based split

df = df.sort_values("time_idx").reset_index(drop=True)

N = len(df)
train_end = int(N * 0.7)
val_end   = int(N * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Total samples:", N)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("time_idx ranges:")
print(" train:", train_df["time_idx"].min(), "->", train_df["time_idx"].max())
print(" val  :", val_df["time_idx"].min(),   "->", val_df["time_idx"].max())
print(" test :", test_df["time_idx"].min(),  "->", test_df["time_idx"].max())


Total samples: 61270
Train: 42889 Val: 9190 Test: 9191
time_idx ranges:
 train: 199 -> 43087
 val  : 43088 -> 52277
 test : 52278 -> 61468


In [8]:
# Cell 7: TimeSeriesDataSet setup (4h target)

MAX_ENCODER_LENGTH = 168    # 7 days of H1
MAX_PREDICTION_LENGTH = 1   # 1 scalar = 4h aggregated return
BATCH_SIZE = 256

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_return_4h",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    target_normalizer=None,         # keep raw log-returns (for live compatibility)
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True,
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True,
)

print("Number of samples:")
print(" training   :", len(training))
print(" validation :", len(validation))
print(" test       :", len(test))


Number of samples:
 training   : 42721
 validation : 9022
 test       : 9023


In [9]:
# Cell 8: Dataloaders

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print("Batches:")
print(" train:", len(train_loader))
print(" val  :", len(val_loader))
print(" test :", len(test_loader))


Batches:
 train: 166
 val  : 36
 test : 36


In [10]:
# Cell 9: Build TFT model (4h move regression)

hidden_size = 32
attention_heads = 4
dropout = 0.10
hidden_continuous_size = 16
lstm_layers = 2
LEARNING_RATE = 1e-3

tft = TFTModel.from_dataset(
    training,
    learning_rate=LEARNING_RATE,

    hidden_size=hidden_size,
    lstm_layers=lstm_layers,
    attention_head_size=attention_heads,
    dropout=dropout,
    hidden_continuous_size=hidden_continuous_size,

    loss=MAE(),            # MAE on 4h log-returns
    output_size=1,         # one 4h move
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("TFT model created.")
print("Number of parameters:", sum(p.numel() for p in tft.parameters()))


TFT model created.
Number of parameters: 128752


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [11]:
# Cell 10: Trainer setup

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="eurusd_h1_tft_4h-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=80,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.1,
    precision=32,         # keep float32 (avoid half precision issues)
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=20,
)

trainer


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [12]:
# Cell 11: Train the 4h TFT

trainer.fit(
    model=tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)


You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 241    | train
3  | prescalers                         | ModuleDict                      | 768    | train
4  | static_variable_selection

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


Epoch 20: 100%|██████████| 166/166 [02:20<00:00,  1.18it/s, v_num=55, train_loss_step=0.0023, val_loss=0.00172, train_loss_epoch=0.00212] 
Best checkpoint path: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_55\checkpoints\eurusd_h1_tft_4h-epoch=12-val_loss=0.001243.ckpt


In [13]:
# Cell 12: Evaluate on test set (4h horizon)

ckpt_path = checkpoint_cb.best_model_path
print("Using checkpoint:", ckpt_path)

best_model = TFTModel.load_from_checkpoint(ckpt_path)

# Predictions
preds_raw = best_model.predict(test_loader)  # [N, 1]
preds = preds_raw.detach().cpu().reshape(-1)

# True 4h targets
all_targets = []
for batch in test_loader:
    x, y_tuple = batch      # (x, (y, weight))
    y = y_tuple[0]
    all_targets.append(y.detach().cpu().reshape(-1))

targets = torch.cat(all_targets)

preds_np = preds.numpy()
targets_np = targets.numpy()

# MAE
mae = np.mean(np.abs(preds_np - targets_np))
print(f"Test MAE (4h log-return units): {mae:.8f}")

# Direction accuracy on 4h move
pred_dir = np.sign(preds_np)
true_dir = np.sign(targets_np)
dir_acc = (pred_dir == true_dir).mean()
print(f"Direction accuracy (4h): {dir_acc:.4f} ({dir_acc*100:.2f}%)")

print("\nSample comparison (first 5):")
for i in range(5):
    print(f"{i}: pred={preds_np[i]:+0.6f}, true={targets_np[i]:+0.6f}")


Using checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_55\checkpoints\eurusd_h1_tft_4h-epoch=12-val_loss=0.001243.ckpt


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\c

Test MAE (4h log-return units): 0.00131796
Direction accuracy (4h): 0.4960 (49.60%)

Sample comparison (first 5):
0: pred=-0.000208, true=-0.000261
1: pred=+0.000125, true=-0.000009
2: pred=-0.000431, true=-0.000410
3: pred=-0.000293, true=-0.000223
4: pred=-0.000143, true=+0.000102


In [14]:
# Cell 13: Copy best checkpoint to a stable name for live usage

import shutil

model_dst = os.path.abspath(
    os.path.join(os.path.dirname(DATA_PATH), "..", "tft_ret_4h.ckpt")
)

print("Source checkpoint:", ckpt_path)
print("Destination      :", model_dst)

shutil.copy2(ckpt_path, model_dst)
print("Checkpoint copied.")


Source checkpoint: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_55\checkpoints\eurusd_h1_tft_4h-epoch=12-val_loss=0.001243.ckpt
Destination      : C:\Users\admin\Desktop\Forex_algo\code\tft_ret_4h.ckpt
Checkpoint copied.
